In [1]:
!pip install contextily

In [2]:
import requests
import json
import os
import re
from pathlib import Path
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import contextily as ctx
from matplotlib.colors import ListedColormap

In [3]:
def fetch_ma_blockgroup_households(acs_year: int = 2023) -> pd.DataFrame:
    """
    Fetch ACS 5-year B11001 estimate+MOE for all Massachusetts (state FIPS 25) block groups.

    Args:
        acs_year: ACS 5-year vintage year (e.g., 2023 for 2019–2023).

    Returns:
        DataFrame with columns: GEOID, NAME, BB110013_001E, B11001_001M.
    """
    base = f"https://api.census.gov/data/{acs_year}/acs/acs5"
    vars_ = ["NAME", "B11001_001E", "B11001_001M"]

    # Get MA counties first, then pull BGs per county (reliable pattern for small geos)
    counties_url = f"{base}?get=NAME&for=county:*&in=state:25"
    counties = requests.get(counties_url, timeout=60)
    counties.raise_for_status()
    county_rows = counties.json()[1:]  # skip header
    county_fips_list = [row[-1] for row in county_rows]

    frames: list[pd.DataFrame] = []
    for county in county_fips_list:
        url = (
            f"{base}"
            f"?get={','.join(vars_)}"
            f"&for=block%20group:*"
            f"&in=state:25%20county:{county}%20tract:*"
        )
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        data = r.json()
        df = pd.DataFrame(data[1:], columns=data[0])

        # Build standard 12-digit BG GEOID: SSCCCTTTTTTB
        df["GEOID"] = df["state"] + df["county"] + df["tract"] + df["block group"]

        # Keep only what we need
        df = df[["GEOID", "NAME", "B11001_001E", "B11001_001M"]]
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)

    # Convert income fields to numeric; blanks become NaN
    for col in ["B11001_001E", "B11001_001M"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out

def fetch_ma_blockgroup_income(acs_year: int = 2023) -> pd.DataFrame:
    """
    Fetch ACS 5-year B19013 estimate+MOE for all Massachusetts (state FIPS 25) block groups.

    Args:
        acs_year: ACS 5-year vintage year (e.g., 2023 for 2019–2023).

    Returns:
        DataFrame with columns: GEOID, NAME, B19013_001E, B19013_001M.
    """
    base = f"https://api.census.gov/data/{acs_year}/acs/acs5"
    vars_ = ["NAME", "B19013_001E", "B19013_001M"]

    # Get MA counties first, then pull BGs per county (reliable pattern for small geos)
    counties_url = f"{base}?get=NAME&for=county:*&in=state:25"
    counties = requests.get(counties_url, timeout=60)
    counties.raise_for_status()
    county_rows = counties.json()[1:]  # skip header
    county_fips_list = [row[-1] for row in county_rows]

    frames: list[pd.DataFrame] = []
    for county in county_fips_list:
        url = (
            f"{base}"
            f"?get={','.join(vars_)}"
            f"&for=block%20group:*"
            f"&in=state:25%20county:{county}%20tract:*"
        )
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        data = r.json()
        df = pd.DataFrame(data[1:], columns=data[0])

        # Build standard 12-digit BG GEOID: SSCCCTTTTTTB
        df["GEOID"] = df["state"] + df["county"] + df["tract"] + df["block group"]

        # Keep only what we need
        df = df[["GEOID", "NAME", "B19013_001E", "B19013_001M"]]
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)

    # Convert income fields to numeric; blanks become NaN
    for col in ["B19013_001E", "B19013_001M"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def download_tiger_ma_blockgroups(tiger_year: int = 2024, out_dir: Path | str = "data") -> Path:
    """
    Download TIGER/Line MA block group shapefile zip and return the local zip path.

    Args:
        tiger_year: TIGER vintage year (e.g., 2024).
        out_dir: Directory to store downloaded files.

    Returns:
        Path to downloaded zip file.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Massachusetts state FIPS = 25
    url = f"https://www2.census.gov/geo/tiger/TIGER{tiger_year}/BG/tl_{tiger_year}_25_bg.zip"
    zip_path = out_dir / f"tl_{tiger_year}_25_bg.zip"

    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    zip_path.write_bytes(resp.content)
    return zip_path


def build_geojson(output_geojson: Path | str = "ma_bg_b19013.geojson") -> None:
    """
    Create a GeoJSON of MA block groups with ACS B19013 attributes.

    Args:
        output_geojson: Output GeoJSON path.
    """
    income = fetch_ma_blockgroup_income(acs_year=2023)
    households = fetch_ma_blockgroup_households(acs_year=2023)

    # Download TIGER BG shapefile
    tiger_zip = download_tiger_ma_blockgroups(tiger_year=2024, out_dir="data")

    # Read shapefile from the downloaded zip
    with zipfile.ZipFile(tiger_zip) as zf:
        # Find the .shp inside the zip
        shp_name = next(n for n in zf.namelist() if n.lower().endswith(".shp"))
        # Extract everything so geopandas/fiona can read sidecar files (.dbf, .shx, etc.)
        extract_dir = tiger_zip.with_suffix("")  # folder next to zip
        if not extract_dir.exists():
            zf.extractall(extract_dir)

    shp_path = next(extract_dir.glob("*.shp"))
    gdf = gpd.read_file(shp_path)

    # TIGER BG files include GEOID; join on that
    gdf = gdf.merge(income, on="GEOID", how="left")
    gdf = gdf.merge(households, on="GEOID", how="left")

    # Write GeoJSON (WGS84 is typical for GeoJSON)
    gdf = gdf.to_crs(epsg=4326)
    gdf.to_file(output_geojson, driver="GeoJSON")


build_geojson("ma_bg_b19013.geojson")
print("Wrote ma_bg_b19013.geojson")

Wrote ma_bg_b19013.geojson


In [4]:
# Configuration
BASE_URL = "https://services5.arcgis.com/lWpwJ2MvpjCmjj94/ArcGIS/rest/services"
SEARCH_TERM = "2026_2029"
OUTPUT_DIR = "gis_downloads"

def safe_filename(name):
    """Clean string to be safe for filenames."""
    return re.sub(r'[\\/*?:"<>|]', "", name).replace(" ", "_")

def get_json(url, params=None):
    """Helper to fetch JSON from the API."""
    if params is None:
        params = {}
    # Only set a default if caller didn't specify format
    params.setdefault("f", "json")

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None

def download_features(layer_url, layer_name, output_path):
    """Downloads all features from a layer using pagination."""
    print(f"  - Downloading layer: {layer_name}...")

    all_features = []
    offset = 0
    batch_size = 2000  # Standard ArcGIS limit

    while True:
        query_params = {
            "where": "1=1",
            "outFields": "*",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
            "f": "geojson"
        }

        # Construct query URL
        query_url = f"{layer_url}/query"
        data = get_json(query_url, query_params)

        if not data or "features" not in data:
            print(f"    Warning: No features found or error for {layer_name}")
            break

        features = data["features"]
        if not features:
            break

        all_features.extend(features)
        print(f"    Fetched {len(features)} features (Total: {len(all_features)})...")

        # Check if we've reached the end
        if "exceededTransferLimit" not in data or not data["exceededTransferLimit"]:
            print(f"    Done with transfer limit")
            break

        print(data["exceededTransferLimit"])

        offset += len(features)

    if all_features:
        # Construct final GeoJSON
        geojson_output = {
            "type": "FeatureCollection",
            "features": all_features
        }

        filename = f"{safe_filename(layer_name)}.geojson"
        filepath = os.path.join(output_path, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(geojson_output, f)
        print(f"  [SUCCESS] Saved {len(all_features)} features to {filepath}")
    else:
        print("  [INFO] Layer was empty.")

def download_geojsons():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    print(f"Checking {BASE_URL} for services matching '{SEARCH_TERM}'...")

    # 1. Get list of services
    catalog = get_json(BASE_URL)
    if not catalog or "services" not in catalog:
        print("Failed to list services. Check URL.")
        return

    services = catalog["services"]
    matched_services = [
        s for s in services if SEARCH_TERM in s["name"]
        or "Berkshire_2026" in s["name"]
        or "Berkshire_2027" in s["name"]
    ]

    print(f"Found {len(matched_services)} matching services: {matched_services}")

    # 2. Process each matching service
    for service in matched_services:
        service_name = service["name"]
        service_url = f"{BASE_URL}/{service_name}/{service['type']}"

        # Handle folder structures if 'name' contains slashes (e.g. Folder/Service)
        # The URL provided seems to be flat, but standard REST APIs use 'name' relative to root.
        # However, 'url' in the service object is usually absolute or relative.
        # Let's rely on constructing it carefully or using the 'url' field if absolute.
        if "url" in service:
            service_url = service["url"]
        else:
             # Fallback construction
             service_url = f"{BASE_URL}/{service_name}/{service['type']}"

        print(f"\nProcessing Service: {service_name}")

        # Get Service Details (to find layers inside)
        service_details = get_json(service_url)
        if not service_details:
            continue

        # 3. Iterate through layers in the service
        layers = service_details.get("layers", [])

        if not layers:
            print("  No layers found in this service.")
            continue

        for layer in layers:
            l_id = layer["id"]
            l_name = layer["name"]

            # Combine service name and layer name for the file
            full_layer_name = f"{service_name}_{l_name}"

            # Construct Layer URL
            layer_url = f"{service_url}/{l_id}"

            print(layer_url)

            download_features(layer_url, full_layer_name, OUTPUT_DIR)

if not os.path.exists(OUTPUT_DIR):
    print(f"Creating output directory: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(os.path.join(OUTPUT_DIR, "Boston_Gas_2026_2029_Sheet1.geojson")):
    print("Downloading...")
    download_geojsons()
else:
    print("GIS files already exist. Skipping download.")

Creating output directory: gis_downloads
Downloading...
Checking https://services5.arcgis.com/lWpwJ2MvpjCmjj94/ArcGIS/rest/services for services matching '2026_2029'...
Found 8 matching services: [{'name': 'Berkshire_2026', 'type': 'FeatureServer', 'url': 'https://services5.arcgis.com/lWpwJ2MvpjCmjj94/ArcGIS/rest/services/Berkshire_2026/FeatureServer'}, {'name': 'Berkshire_2027', 'type': 'FeatureServer', 'url': 'https://services5.arcgis.com/lWpwJ2MvpjCmjj94/ArcGIS/rest/services/Berkshire_2027/FeatureServer'}, {'name': 'Boston_Gas_2026_2029', 'type': 'FeatureServer', 'url': 'https://services5.arcgis.com/lWpwJ2MvpjCmjj94/ArcGIS/rest/services/Boston_Gas_2026_2029/FeatureServer'}, {'name': 'Colonial_Gas_2026_2029', 'type': 'FeatureServer', 'url': 'https://services5.arcgis.com/lWpwJ2MvpjCmjj94/ArcGIS/rest/services/Colonial_Gas_2026_2029/FeatureServer'}, {'name': 'EGMA_2026_2029', 'type': 'FeatureServer', 'url': 'https://services5.arcgis.com/lWpwJ2MvpjCmjj94/ArcGIS/rest/services/EGMA_2026_20

In [5]:
# Download municipality layers
# Configuration
PORTAL_SHARING_ROOT = "https://mass-eoeea.maps.arcgis.com/sharing/rest"
ITEM_ID = "9fb7faa26fe849888422a58deea1f776"
OUTPUT_DIR = "gis_downloads"
BATCH_SIZE = 2000  # Standard ArcGIS limit


def safe_filename(name):
    """Clean string to be safe for filenames."""
    return re.sub(r'[\\/*?:"<>|]', "", name).replace(" ", "_")


def get_json(url, params=None):
    """Helper to fetch JSON from the ArcGIS REST API."""
    if params is None:
        params = {}
    params.setdefault("f", "json")

    # Optional token for private items
    token = os.environ.get("ARCGIS_TOKEN")
    if token and "token" not in params:
        params["token"] = token

    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        if isinstance(data, dict) and data.get("error"):
            print(f"ArcGIS error for {url}: {data['error']}")
            return None
        return data
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None


def find_feature_service_url(item_id):
    """Resolve the FeatureServer URL from an ArcGIS Online item."""
    item_url = f"{PORTAL_SHARING_ROOT}/content/items/{item_id}"
    item_json = get_json(item_url)
    if not item_json:
        return None

    # Most Feature Layer items expose the service directly here
    if "url" in item_json and item_json["url"]:
        return item_json["url"].rstrip("/")

    # Fallback: inspect item data
    item_data = get_json(f"{item_url}/data")
    if not item_data:
        return None

    for key in ("url", "serviceUrl", "serviceURL"):
        if key in item_data and item_data[key]:
            return str(item_data[key]).rstrip("/")

    return None


def download_features(layer_url, layer_name, output_path):
    """Download all features from a layer using pagination (GeoJSON)."""
    print(f"  - Downloading layer: {layer_name}...")

    all_features = []
    offset = 0

    while True:
        query_params = {
            "where": "1=1",
            "outFields": "*",
            "resultOffset": offset,
            "resultRecordCount": BATCH_SIZE,
            "f": "geojson",
        }

        query_url = f"{layer_url}/query"
        data = get_json(query_url, query_params)

        if not data or "features" not in data:
            print(f"    Warning: No features found or error for {layer_name}")
            break

        features = data["features"]
        if not features:
            break

        all_features.extend(features)
        print(f"    Fetched {len(features)} features (Total: {len(all_features)})...")

        if not data.get("exceededTransferLimit", False):
            break

        offset += len(features)

    if all_features:
        geojson_output = {
            "type": "FeatureCollection",
            "features": all_features,
        }

        filename = f"{safe_filename(layer_name)}.geojson"
        filepath = os.path.join(output_path, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(geojson_output, f)

        print(f"  [SUCCESS] Saved {len(all_features)} features to {filepath}")
    else:
        print("  [INFO] Layer was empty.")

if not os.path.exists(OUTPUT_DIR):
    print(f"Creating output directory: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Resolving FeatureServer from item...")
service_url = find_feature_service_url(ITEM_ID)
print(f"Feature service: {service_url}")

service_details = get_json(service_url)
layers = service_details.get("layers", [])

if not layers:
    raise RuntimeError("No layers found in FeatureServer")

# Prefer a municipalities layer; otherwise use first layer
layer = None
for lyr in layers:
    name = lyr.get("name", "").lower()
    if "municip" in name or "town" in name or "city" in name:
        layer = lyr
        break

if layer is None:
    layer = layers[0]

layer_id = layer["id"]
layer_name = layer["name"]
layer_url = f"{service_url}/{layer_id}"

output_file = os.path.join(
    OUTPUT_DIR, f"{safe_filename(layer_name)}.geojson"
)

if os.path.exists(output_file):
    print(f"Output already exists, skipping download: {output_file}")
else:
    print(f"Selected layer: {layer_name}")
    download_features(layer_url, layer_name, OUTPUT_DIR)

Resolving FeatureServer from item...
Feature service: https://arcgisserver.digital.mass.gov/arcgisserver/rest/services/AGOL/Towns_survey_polym/FeatureServer
Selected layer: Massachusetts Municipalities
  - Downloading layer: Massachusetts Municipalities...
    Fetched 351 features (Total: 351)...
  [SUCCESS] Saved 351 features to gis_downloads/Massachusetts_Municipalities.geojson


In [6]:
# Download MA census block groups
# Configuration
PORTAL_SHARING_ROOT = "https://massgis.maps.arcgis.com/sharing/rest"
ITEM_ID = "b05fdf4d38434e0793838513642789b0"
OUTPUT_DIR = "gis_downloads"
BATCH_SIZE = 2000  # Standard ArcGIS limit


def safe_filename(name: str) -> str:
    """Clean a string to be safe for filenames."""
    return re.sub(r'[\\/*?:"<>|]', "", name).replace(" ", "_")


def get_json(url: str, params: dict | None = None) -> dict | None:
    """Fetch JSON from the ArcGIS REST API (adds f=json and optional ARCGIS_TOKEN)."""
    if params is None:
        params = {}
    params.setdefault("f", "json")

    token = os.environ.get("ARCGIS_TOKEN")
    if token and "token" not in params:
        params["token"] = token

    try:
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, dict) and data.get("error"):
            print(f"ArcGIS error for {url}: {data['error']}")
            return None
        return data
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None


def find_feature_service_url(item_id: str) -> str | None:
    """Resolve the FeatureServer URL from an ArcGIS Online item."""
    item_url = f"{PORTAL_SHARING_ROOT}/content/items/{item_id}"
    item_json = get_json(item_url)
    if not item_json:
        return None

    if item_json.get("url"):
        return str(item_json["url"]).rstrip("/")

    item_data = get_json(f"{item_url}/data")
    if not item_data:
        return None

    for key in ("url", "serviceUrl", "serviceURL"):
        if item_data.get(key):
            return str(item_data[key]).rstrip("/")

    return None


def download_features(layer_url: str, layer_name: str, output_path: str) -> None:
    """Download all features from a layer using pagination (GeoJSON)."""
    print(f"  - Downloading layer: {layer_name}...")

    all_features: list[dict] = []
    offset = 0

    while True:
        query_params = {
            "where": "1=1",
            "outFields": "*",
            "resultOffset": offset,
            "resultRecordCount": BATCH_SIZE,
            "f": "geojson",
        }
        data = get_json(f"{layer_url}/query", query_params)

        if not data or "features" not in data:
            print(f"    Warning: No features found or error for {layer_name}")
            break

        features = data["features"] or []
        if not features:
            break

        all_features.extend(features)
        print(f"    Fetched {len(features)} features (Total: {len(all_features)})...")

        if not data.get("exceededTransferLimit", False):
            break

        offset += len(features)

    if not all_features:
        print("  [INFO] Layer was empty.")
        return

    out_geojson = {"type": "FeatureCollection", "features": all_features}
    filename = f"{safe_filename(layer_name)}.geojson"
    filepath = os.path.join(output_path, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(out_geojson, f)

    print(f"  [SUCCESS] Saved {len(all_features)} features to {filepath}")


# --- Main ---
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Resolving FeatureServer from item...")
service_url = find_feature_service_url(ITEM_ID)
if not service_url:
    raise RuntimeError("Could not resolve FeatureServer URL from item ID")

print(f"Feature service: {service_url}")

service_details = get_json(service_url)
if not service_details:
    raise RuntimeError("Could not read FeatureServer details")

layers = service_details.get("layers", [])
if not layers:
    raise RuntimeError("No layers found in FeatureServer")

# Prefer a block groups layer; otherwise use first layer
layer = None
for lyr in layers:
    name = lyr.get("name") or ""
    if name == "Census 2020 Block Groups":
        layer = lyr
        break
layer = layer or layers[0]

layer_id = layer["id"]
layer_name = layer["name"]
layer_url = f"{service_url}/{layer_id}"

output_file = os.path.join(OUTPUT_DIR, f"{safe_filename(layer_name)}.geojson")
if os.path.exists(output_file):
    print(f"Output already exists, skipping download: {output_file}")
else:
    print(f"Selected layer: {layer_name} (id={layer_id})")
    download_features(layer_url, layer_name, OUTPUT_DIR)

Resolving FeatureServer from item...
Feature service: https://arcgisserver.digital.mass.gov/arcgisserver/rest/services/AGOL/Census2020_Base_Geography/MapServer
Selected layer: Census 2020 Block Groups (id=2)
  - Downloading layer: Census 2020 Block Groups...
    Fetched 1000 features (Total: 1000)...
    Fetched 1000 features (Total: 2000)...
    Fetched 1000 features (Total: 3000)...
    Fetched 1000 features (Total: 4000)...
    Fetched 1000 features (Total: 5000)...
    Fetched 109 features (Total: 5109)...
  [SUCCESS] Saved 5109 features to gis_downloads/Census_2020_Block_Groups.geojson


In [7]:
filenames = os.listdir(OUTPUT_DIR)
dfs = []

company_mappings = {
    "EGMA": "Eversource",
    "Boston": "National Grid",
    "Colonial": "National Grid"
}

for filename in filenames:
    if filename.endswith('.geojson') and ("2026" in filename or "2027" in filename):
        company = filename.split('_')[0]
        filepath = os.path.join(OUTPUT_DIR, filename)
        df = gpd.read_file(filepath)
        company = company_mappings.get(company, company)
        df["LDC"] = company
        print(f"Processing {company}: {df.shape} records")
        dfs.append(df)

gsep_df = pd.concat(dfs, ignore_index=True)
print(f"Total: {gsep_df.shape} rows x columns")
gsep_df.head()

Processing National Grid: (1238, 19) records
Processing Eversource: (398, 14) records
Processing Liberty: (292, 73) records
Processing Berkshire: (34, 30) records
Processing Unitil: (230, 32) records
Processing Berkshire: (37, 30) records
Processing Eversource: (1445, 14) records
Processing National Grid: (100, 19) records
Total: (3774, 138) rows x columns


/tmp/ipython-input-1134982702.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gsep_df = pd.concat(dfs, ignore_index=True)


,Line,Company,Division,Town,Town_Code,State,Description__street_segment_,Diameter,Material,Operating_Pressure,...,No_of_Services,Total_Meter_Count,RISK_MODEL_SCORE,Project_Driver__Paving__Encroac,Project_Comments_Justification,Total_Cost_,Retirmement_footage,Installation_footage,F_LPP_Footage_,F_LPP_Mileage__
0,40.0,Boston Gas,Beverly,Lynn,LNN,MA,"9-87 DEN QUARRY RD, LNN, & 7-21 CANNON VIEW CIR",4.0,Cast Iron,LP to 60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40.0,Boston Gas,Beverly,Lynn,LNN,MA,"9-87 DEN QUARRY RD, LNN, & 7-21 CANNON VIEW CIR",4.0,Cast Iron,LP to 60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41.0,Boston Gas,Beverly,Lynnfield,LNF,MA,"14-23 THOMAS RD, LNF, & 16-21 WILLIAMS RD",2.0,Steel-WI,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41.0,Boston Gas,Beverly,Lynnfield,LNF,MA,"14-23 THOMAS RD, LNF, & 16-21 WILLIAMS RD",2.0,Steel-WI,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,42.0,Boston Gas,Beverly,Marblehead,MAR,MA,"1-11 ROLLESTON RD, MAR",4.0,Cast Iron,LP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
ejc_df = gpd.read_file("ej_gis/EJ_POLY.shp")

DataSourceError: ej_gis/EJ_POLY.shp: No such file or directory

In [ ]:
# Overlay with EJCs
fig, ax = plt.subplots(1, 1, figsize=(15, 15), frameon=False)

# Plot EJC boundaries
ejc_df.to_crs(epsg=3857).plot(ax=ax, color='#00ff0044', edgecolor='none', label='EJCs')

# Overlay GSEP projects
gsep_df.to_crs(epsg=3857).plot(column='LDC', legend=True, ax=ax, markersize=1)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title('Planned 2026-2029 GSEP projects by LDC')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
# Ensure both GeoDataFrames have the same CRS for accurate spatial operations
gsep_df_reprojected = gsep_df.to_crs(ejc_df.crs)

# Perform a spatial join to find points within EJC polygons
# 'inner' join type keeps only points from gsep_df_reprojected that intersect with ejc_df
points_in_ejc = gpd.sjoin(gsep_df_reprojected, ejc_df, how="inner", predicate="intersects")

# Create a boolean column indicating if a point is within an EJC
gsep_df_reprojected['is_in_ejc'] = gsep_df_reprojected.index.isin(points_in_ejc.index)

# Count the number of points in each category
counts = gsep_df_reprojected['is_in_ejc'].value_counts()

# Map boolean values to descriptive strings for better plotting labels
counts.index = counts.index.map({True: 'In EJC', False: 'Not in EJC'})

# Calculate percentages
total_projects = counts.sum()
percentages = (counts / total_projects * 100).round(1)

# Create the bar chart
sns.set("talk", style="white")
plt.figure(figsize=(8, 6))
ax = counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('Location of planned GSEP projects (2026-2029)')
plt.ylabel('Number of GSEP projects')
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels to the bars
for i, bar in enumerate(ax.patches):
    percentage_value = percentages.iloc[i]
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 200,
            f'{percentage_value}%', ha='center', va='bottom')

plt.show()

In [ ]:
# Load municipal and census block layers
muni_df = gpd.read_file("gis_downloads/Massachusetts_Municipalities.geojson")
block_df = gpd.read_file("gis_downloads/Census_2020_Block_Groups.geojson")

# Merge in annual electric usage by municipality
municipal_usage = pd.read_csv("2023_municipal_usage_data.csv")
municipal_usage["Annual Electric Usage (MWh)"] = pd.to_numeric(municipal_usage["Annual Electric Usage (MWh)"], errors="coerce").fillna(0)
municipal_usage["Annual Gas Usage (Therms)"] = pd.to_numeric(municipal_usage["Annual Gas Usage (Therms)"], errors="coerce").fillna(0)
municipal_usage["Annual Energy Usage (MWh)"] = municipal_usage["Annual Electric Usage (MWh)"] + 0.0293001 * municipal_usage["Annual Gas Usage (Therms)"]
municipal_usage = municipal_usage[["Municipality", "Annual Electric Usage (MWh)", "Annual Energy Usage (MWh)"]]
municipal_usage["Municipality"] = municipal_usage["Municipality"].str.upper()
muni_df = muni_df.merge(municipal_usage, left_on="TOWN", right_on="Municipality", how="left").drop(columns=["Municipality"])

In [ ]:
# Get electricity use by EJCs vs Non-EJCs

# First, label census blocks as EJC or non EJC
ejc_centroids = ejc_df.copy()
ejc_centroids["geometry"] = ejc_df.centroid
ejc_centroids = ejc_centroids.to_crs(block_df.crs)

ejc_block_indices = gpd.sjoin(
    left_df=ejc_centroids,
    right_df=block_df,
    how="inner",
    predicate="within"
).index_right
block_df["is_ejc"] = False
block_df.loc[ejc_block_indices, "is_ejc"] = True

# Then get the fraction of EJC census blocks in each municipality
block_muni = gpd.sjoin(
    left_df=block_df,
    right_df=muni_df,
    how="inner",
    predicate="intersects"
)
muni_ejc_stats = block_muni.groupby("index_right").agg(
    total_blocks=("is_ejc", "count"),
    ejc_blocks=("is_ejc", "sum")
)
muni_ejc_stats["ejc_fraction"] = (
    muni_ejc_stats["ejc_blocks"] / muni_ejc_stats["total_blocks"]
)
muni_df = muni_df.join(muni_ejc_stats[["ejc_fraction"]], how="left")
muni_df["ejc_fraction"] = muni_df["ejc_fraction"].fillna(0)

# Compute EJC and non-EJC usage
ejc_usage = muni_df["Annual Energy Usage (MWh)"] * muni_df["ejc_fraction"]
non_ejc_usage = muni_df["Annual Energy Usage (MWh)"] * (1 - muni_df["ejc_fraction"])
total_usage = ejc_usage + non_ejc_usage

In [ ]:
# Create the bar chart
plot_df = pd.DataFrame({
    'Category': ['In EJC', 'Not in EJC'],
    'Usage (%)': [
        ejc_usage.sum() / total_usage.sum() * 100,
        non_ejc_usage.sum() / total_usage.sum() * 100
    ]
})

sns.set_context("talk")
sns.set_style("white")
fig, ax = plt.subplots(figsize=(8, 6))

ax.bar(plot_df['Category'], plot_df['Usage (%)'],
       color=['skyblue', 'lightcoral'])
ax.set_title('2023 Energy Consumption (% of statewide)')
ax.set_ylabel('%')
ax.set_xlabel('')
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels
for i, (cat, val) in enumerate(zip(plot_df['Category'], plot_df['Usage (%)'])):
    ax.text(i, val - 5, f'{val:.1f}%', ha='center', va='bottom')

plt.tight_layout()
# plt.show()

In [ ]:
# Load national grid distribution circuit map and list of NWA opportunities
filenames = os.listdir("national_grid")
national_grid_dfs = []

for filename in filenames:
    if filename.endswith('.geojson'):
        filepath = os.path.join("national_grid", filename)
        df = gpd.read_file(filepath)
        national_grid_dfs.append(df)

national_grid_df = pd.concat(national_grid_dfs, ignore_index=True)

nwa_feeders = list(set([
  "05_01_26W2",
  "05_01_304W2",
  "05_01_304W3",
  "05_01_304W4",
  "05_01_304W5",
  "05_01_304W6",
  "05_01_26W3",
  "05_01_406L2",
  "05_01_406L4",
  "05_01_412L1",
  "05_01_412L3",
  "05_01_412L6",
  "05_01_415L2",
  "05_01_406L1",
  "05_01_406L3",
  "05_01_413L4",
  "05_01_415L1",
  "05_01_415L3",
  "05_05_3451W2",
  "05_07_912W21",
  "05_07_797W1",
  "05_07_797W20",
  "05_07_912W22",
  "05_07_912W55",
  "05_07_912W73",
  "05_07_912W74",
  "05_07_912W75",
  "05_14_75L2",
  "05_12_11J13",
  "05_09_523L4",
  "05_09_501L2",
  "05_09_523L1",
  "05_09_523L2",
  "05_01_4J324",
  "05_01_8J364",
  "05_01_9J329",
  "05_01_3J341",
  "05_01_3J372",
  "05_01_3J373",
  "05_09_702W2",
  "05_09_702W1",
  "05_09_702W3",
  "05_09_705W3",
  "05_07_93W42",
  "05_07_93W43",
  "05_07_797W24",
  "05_07_797W29",
  "05_07_95W3",
  "05_07_99W62",
  "05_07_91W47",
  "05_07_91W41",
  "05_07_911W77",
  "05_12_67J4",
  "05_12_67J1",
  "05_12_5C3",
  "05_12_5J10",
  "05_12_5C1",
  "05_09_503L1",
  "05_09_503L2",
  "05_09_503L4",
  "05_09_514L1",
  "05_09_139L1",
  "05_09_139L3",
  "05_09_139L5",
  "05_09_507L2",
  "05_09_508L4",
  "05_12_21J30",
  "05_12_21J21",
  "05_12_21J25",
  "05_12_21J32",
  "05_12_21J23",
  "05_12_3J903",
  "05_01_HT52",
  "05_01_HT45",
  "04_04_101L2",
  "04_04_101L4",
  "04_04_101L6",
  "04_04_101L8",
  "05_05_3431W1",
  "05_05_3431W2",
  "05_05_3432W1",
  "05_05_3432W2",
  "05_05_3424W1",
  "05_05_3424W3",
  "05_05_3424W5",
  "05_05_349W1",
  "05_12_3J6",
  "05_12_1J12",
  "05_12_3J6",
  "05_12_49W6",
  "05_12_29W3",
  "05_12_3J2",
  "05_12_3J3",
  "05_12_3J4",
  "05_07_75W5",
  "05_07_75W3",
  "05_07_75W1",
  "05_07_75W7",
  "05_07_97W5"
]))

national_grid_df["nwa_opportunity"] = national_grid_df["Master_CDF"].isin(nwa_feeders)

In [ ]:
len(nwa_feeders)

In [ ]:
# Overlay with EJCs
fig, ax = plt.subplots(1, 1, figsize=(15, 15), frameon=False)

# Plot EJC boundaries
ejc_df.to_crs(epsg=3857).plot(ax=ax, color='#00ff0044', edgecolor='none', label='EJCs')

# Overlay GSEP projects
national_grid_df[national_grid_df["nwa_opportunity"]].to_crs(epsg=3857).plot(legend=False, ax=ax, markersize=1, linewidth=0.5)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title('Announced 2026-2029 NWA opportunities (National Grid)')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
# Figure out how much of each feeder is in an EJC
national_grid_df["length_in_ejc"] = (
    national_grid_df.to_crs(epsg=26986).geometry
        .intersection(ejc_df.to_crs(epsg=26986).geometry.union_all())
        .length
        .fillna(0.0)
)

In [ ]:
# Sum the length of projects in EJC vs non EJC regions
nwa_length_in_ejcs = national_grid_df[national_grid_df["nwa_opportunity"]]["length_in_ejc"].sum()
nwa_total_length = national_grid_df[national_grid_df["nwa_opportunity"]].to_crs(epsg=26986).geometry.length.sum()
nwa_length_outside_ejcs = nwa_total_length - nwa_length_in_ejcs

counts = pd.Series({
    'In EJC': nwa_length_in_ejcs / 1e3 * 0.621371,
    'Not in EJC': nwa_length_outside_ejcs / 1e3 * 0.621371
})

# Calculate percentages
total_projects = counts.sum()
percentages = (counts / total_projects * 100).round(1)

# Create the bar chart
sns.set("talk", style="white")
plt.figure(figsize=(8, 6))
ax = counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('National Grid announced NWA opportunities (2026-2029)')
plt.ylabel('Miles of feeder')
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels to the bars
for i, bar in enumerate(ax.patches):
    percentage_value = percentages.iloc[i]
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 200,
            f'{percentage_value}%', ha='center', va='bottom')

plt.show()

In [ ]:
# Load median income dataset
median_income_df = gpd.read_file("ma_bg_b19013.geojson")

# Drop missing rows
median_income_df = median_income_df[median_income_df["B19013_001E"].notna() & (median_income_df["B19013_001E"] > 0)]

# Filter to LMI regions
MA_MEDIAN_INCOME_2020 = 84_385
threshold = 0.8
median_income_df["is_lmi"] = median_income_df["B19013_001E"] <= MA_MEDIAN_INCOME_2020 * threshold
lmi_df = median_income_df[median_income_df["is_lmi"]]

In [ ]:
# Overlay GSEP and LMI block groups
fig, ax = plt.subplots(1, 1, figsize=(15, 15), frameon=False)

# Plot EJC boundaries
lmi_df.to_crs(epsg=3857).plot(ax=ax, color='#9448BC44', edgecolor='none', label='LMI Census Block Groups')

# Overlay GSEP projects
gsep_df.to_crs(epsg=3857).plot(column='LDC', legend=True, ax=ax, markersize=1)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title('Planned 2026-2029 GSEP projects by LDC')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
# Ensure both GeoDataFrames have the same CRS for accurate spatial operations
gsep_df_reprojected = gsep_df_reprojected.to_crs(lmi_df.crs)

# Perform a spatial join to find points within LMI polygons
# 'inner' join type keeps only points from gsep_df_reprojected that intersect with lmi_df
points_in_lmi = gpd.sjoin(gsep_df_reprojected, lmi_df, how="inner", predicate="intersects")

# Create a boolean column indicating if a point is within an LMI
gsep_df_reprojected['is_in_lmi'] = gsep_df_reprojected.index.isin(points_in_lmi.index)

# Count the number of points in each category
counts = gsep_df_reprojected['is_in_lmi'].value_counts()

# Map boolean values to descriptive strings for better plotting labels
counts.index = counts.index.map({True: 'In LMI', False: 'Not in LMI'})

# Re-order to have "in LMI" first
counts = counts.loc[["In LMI", "Not in LMI"]]

# Calculate percentages
total_projects = counts.sum()
percentages = (counts / total_projects * 100).round(1)

# Create the bar chart
sns.set("talk", style="white")
plt.figure(figsize=(8, 6))
ax = counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('Location of planned GSEP projects (2026-2029)')
plt.ylabel('Number of GSEP projects')
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels to the bars
for i, bar in enumerate(ax.patches):
    percentage_value = percentages.iloc[i]
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 200,
            f'{percentage_value}%', ha='center', va='bottom')

plt.show()

In [ ]:
# Figure out how much of each feeder is in an EJC
national_grid_df["length_in_lmi"] = (
    national_grid_df.to_crs(epsg=26986).geometry
        .intersection(lmi_df.to_crs(epsg=26986).geometry.union_all())
        .length
        .fillna(0.0)
)

In [ ]:
# Overlay with LMI
fig, ax = plt.subplots(1, 1, figsize=(15, 15), frameon=False)

# Plot EJC boundaries
lmi_df.to_crs(epsg=3857).plot(ax=ax, color='#9448BC44', edgecolor='none', label='LMI')

# Overlay GSEP projects
national_grid_df[national_grid_df["nwa_opportunity"]].to_crs(epsg=3857).plot(legend=False, ax=ax, markersize=1, linewidth=0.5)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title('Announced 2026-2029 NWA opportunities (National Grid)')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
# Sum the length of projects in LMI vs non LMI regions
nwa_length_in_lmis = national_grid_df[national_grid_df["nwa_opportunity"]]["length_in_lmi"].sum()
nwa_total_length = national_grid_df[national_grid_df["nwa_opportunity"]].to_crs(epsg=26986).geometry.length.sum()
nwa_length_outside_lmis = nwa_total_length - nwa_length_in_lmis

counts = pd.Series({
    'In LMI': nwa_length_in_lmis / 1e3 * 0.621371,
    'Not in LMI': nwa_length_outside_lmis / 1e3 * 0.621371
})

# Calculate percentages
total_projects = counts.sum()
percentages = (counts / total_projects * 100).round(1)

# Create the bar chart
sns.set("talk", style="white")
plt.figure(figsize=(8, 6))
ax = counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('National Grid announced NWA opportunities (2026-2029)')
plt.ylabel('Miles of feeder')
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels to the bars
for i, bar in enumerate(ax.patches):
    percentage_value = percentages.iloc[i]
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 200,
            f'{percentage_value}%', ha='center', va='bottom')

plt.show()

In [ ]:
median_income_df.groupby("is_lmi")["B11001_001E"].sum() / median_income_df["B11001_001E"].sum()

In [ ]:
# Get electricity use by LMI vs Non-LMI

# First, label census blocks as LMI or non LMI
lmi_centroids = lmi_df.copy()
lmi_centroids["geometry"] = lmi_df.centroid
lmi_centroids = lmi_centroids.to_crs(block_df.crs)

lmi_block_indices = gpd.sjoin(
    left_df=lmi_centroids,
    right_df=block_df,
    how="inner",
    predicate="within"
).index_right
block_df["is_lmi"] = False
block_df.loc[lmi_block_indices, "is_lmi"] = True

# Then get the fraction of EJC census blocks in each municipality
block_muni = gpd.sjoin(
    left_df=block_df,
    right_df=muni_df,
    how="inner",
    predicate="intersects"
)
muni_lmi_stats = block_muni.groupby("index_right").agg(
    total_blocks=("is_lmi", "count"),
    lmi_blocks=("is_lmi", "sum")
)
muni_lmi_stats["lmi_fraction"] = (
    muni_lmi_stats["lmi_blocks"] / muni_lmi_stats["total_blocks"]
)
muni_df = muni_df.join(muni_lmi_stats[["lmi_fraction"]], how="left")
muni_df["lmi_fraction"] = muni_df["lmi_fraction"].fillna(0)

# Compute EJC and non-EJC usage
lmi_usage = muni_df["Annual Energy Usage (MWh)"] * muni_df["lmi_fraction"]
non_lmi_usage = muni_df["Annual Energy Usage (MWh)"] * (1 - muni_df["lmi_fraction"])
total_usage = lmi_usage + non_lmi_usage

print(lmi_usage.sum() / total_usage.sum() * 100)
print(non_lmi_usage.sum() / total_usage.sum() * 100)

In [ ]:
# Load gateway cities
gateway_cities_df = gpd.read_file("Massachusetts_Gateway_Cities.geojson")

gateway_cities_df = gateway_cities_df[gateway_cities_df["GATEWAY"] == "Y"].copy()

In [ ]:
# Overlay with Gateway cities
fig, ax = plt.subplots(1, 1, figsize=(15, 15), frameon=False)

gateway_cities_df.to_crs(epsg=3857).plot(ax=ax, color='#a7260844', edgecolor='none', label='Gateway Cities')
gsep_df.to_crs(epsg=3857).plot(column='LDC', legend=True, ax=ax, markersize=1)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title('Planned 2026-2029 GSEP projects by LDC')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15), frameon=False)

gateway_cities_df.to_crs(epsg=3857).plot(ax=ax, color='#a7260844', edgecolor='none', label='LMI')
national_grid_df[national_grid_df["nwa_opportunity"]].to_crs(epsg=3857).plot(legend=False, ax=ax, markersize=1, linewidth=0.5)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title('Announced 2026-2029 NWA opportunities (National Grid)')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
print("% of MA land area in Gateway Cities:")
print(gateway_cities_df.to_crs(epsg=26986).geometry.area.sum() / block_df.to_crs(epsg=26986).geometry.area.sum() * 100)

In [ ]:
muni_df.merge(
    gateway_cities_df[["TOWN", "GATEWAY"]],
    how="left"
).fillna("N").groupby("GATEWAY")["Annual Energy Usage (MWh)"].sum() / muni_df["Annual Energy Usage (MWh)"].sum() * 100

In [ ]:
muni_df.merge(
    gateway_cities_df[["TOWN", "GATEWAY"]],
    how="left"
).fillna("N").groupby("GATEWAY")["POP2020"].sum() / muni_df["POP2020"].sum() * 100

In [ ]:
# Ensure both GeoDataFrames have the same CRS for accurate spatial operations
gsep_df_reprojected = gsep_df_reprojected.to_crs(gateway_cities_df.crs)

# Perform a spatial join to find points within Gateway City polygons
# 'inner' join type keeps only points from gsep_df_reprojected that intersect with gateway_cities_df
points_in_gateway = gpd.sjoin(gsep_df_reprojected, gateway_cities_df, how="inner", predicate="intersects")

# Create a boolean column indicating if a point is within an Gateway City
gsep_df_reprojected['is_in_gateway'] = gsep_df_reprojected.index.isin(points_in_gateway.index)

# Count the number of points in each category
counts = gsep_df_reprojected['is_in_gateway'].value_counts()

# Map boolean values to descriptive strings for better plotting labels
counts.index = counts.index.map({True: 'In Gateway City', False: 'Not in Gateway City'})

# Re-order to have "in Gateway City" first
counts = counts.loc[["In Gateway City", "Not in Gateway City"]]

# Calculate percentages
total_projects = counts.sum()
percentages = (counts / total_projects * 100).round(1)

# Create the bar chart
sns.set("talk", style="white")
plt.figure(figsize=(8, 6))
ax = counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('Location of planned GSEP projects (2026-2029)')
plt.ylabel('Number of GSEP projects')
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels to the bars
for i, bar in enumerate(ax.patches):
    percentage_value = percentages.iloc[i]
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 200,
            f'{percentage_value}%', ha='center', va='bottom')

plt.show()

In [ ]:
# Figure out how much of each feeder is in a gateway city
national_grid_df["length_in_gateway_cities"] = (
    national_grid_df.to_crs(epsg=26986).geometry
        .intersection(gateway_cities_df.to_crs(epsg=26986).geometry.union_all())
        .length
        .fillna(0.0)
)

In [ ]:
# Sum the length of projects in Gateway vs non Gateway regions LMI
nwa_length_in_gateway_cities = national_grid_df[national_grid_df["nwa_opportunity"]]["length_in_gateway_cities"].sum()
nwa_total_length = national_grid_df[national_grid_df["nwa_opportunity"]].to_crs(epsg=26986).geometry.length.sum()
nwa_length_outside_gateway_cities = nwa_total_length - nwa_length_in_gateway_cities

counts = pd.Series({
    'In Gateway City': nwa_length_in_gateway_cities / 1e3 * 0.621371,
    'Not in Gateway City': nwa_length_outside_gateway_cities / 1e3 * 0.621371
})

# Calculate percentages
total_projects = counts.sum()
percentages = (counts / total_projects * 100).round(1)

# Create the bar chart
sns.set("talk", style="white")
plt.figure(figsize=(8, 6))
ax = counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('National Grid announced NWA opportunities (2026-2029)')
plt.ylabel('Miles of feeder')
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentage labels to the bars
for i, bar in enumerate(ax.patches):
    percentage_value = percentages.iloc[i]
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 200,
            f'{percentage_value}%', ha='center', va='bottom')

plt.show()

In [ ]:
# Plot a heatmap of how many regions a GSEP project overlaps with
gsep_df_reprojected["equity_region_overlaps"] = (
    gsep_df_reprojected["is_in_ejc"].astype(int)
    + gsep_df_reprojected["is_in_lmi"].astype(int)
    + gsep_df_reprojected["is_in_gateway"].astype(int)
)
fig, ax = plt.subplots(1, 1, figsize=(15, 15), frameon=False)

cats = [0, 1, 2, 3]
cmap = ListedColormap(["#FFFFFF", "#CAD2DD", "#00A2FF", "#00111B"])

gsep_df_reprojected.to_crs(epsg=3857).plot(
    column="equity_region_overlaps",
    categorical=True,
    categories=cats,
    cmap=cmap,
    legend=True,
    ax=ax,
    markersize=2,
)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_title('Planned 2026-2029 GSEP projects by overlap with equity regions')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
# Set style to match existing plot
sns.set("talk", style="white")
plt.figure(figsize=(8, 6))

# Compute counts and percentages
counts = (
    gsep_df_reprojected["equity_region_overlaps"]
    .value_counts()
    .sort_index()
)
percentages = counts / counts.sum() * 100

# Plot histogram
hist = plt.hist(
    gsep_df_reprojected["equity_region_overlaps"],
    bins=[-0.5, 0.5, 1.5, 2.5, 3.5],
)

# Overlay percentage labels
for x, count in counts.items():
    pct = percentages.loc[x]
    plt.text(
        x,
        count-100,
        f"{pct:.0f}%",
        ha="center",
        va="bottom",
        color="white",
    )

plt.title("Planned GSEP projects (2026–2029)")
plt.ylabel("Number of projects")
plt.xlabel("Number of equity regions overlapped")
plt.xticks([0, 1, 2, 3])
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.show()